In [37]:
import pandas as pd, matplotlib.pyplot as plt ;
from matplotlib.backends.backend_pdf import PdfPages

s, p = pd.read_csv('output/sales_transactions_cleaned.csv') , pd.read_csv('products.csv')

p['cost'], p['price'], p['category'] = pd.to_numeric(p['cost'].astype(str).str.replace('$','', regex = False), errors = 'coerce'), pd.to_numeric(p['price'].astype(str).str.replace('$','', regex = False), errors = 'coerce'), p['category'].replace({'Pastry':'Pastries'})

total_q = s.assign(revenue = (s.quantity * s.price) - s.discount_amount.fillna(0)).merge(p[['product_id', 'product_name', 'category', 'cost']], on = 'product_id', how = 'left')

top3 = total_q.groupby('product_name')[['quantity', 'revenue']].sum().nlargest(3, 'quantity').reset_index().rename(columns = {'quantity':'Total Quantity Sold', 'revenue' : 'Total Revenue'})
top3['Total Revenue'] = top3['Total Revenue'].apply(lambda x: f"${x:,.2f}")
top3['Total Quantity Sold'] = top3['Total Quantity Sold'].apply(lambda v: f"{v:,.0f}")
cr = total_q[total_q.category.isin(['Pastries', 'Bread', 'Tarte'])].groupby('category')['revenue'].sum()

with PdfPages('output/Session1_ProductPerformance.pdf') as pdf:
    fig, ax = plt.subplots(); cr.plot(kind = 'bar', color = 'red', ax = ax, title = 'Total Revenue by Category', ylabel = 'Total Revenue ($)', rot = 0) ;
    pdf.savefig(fig, bbox_inches = 'tight');
    plt.close()
    fig, ax = plt.subplots(figsize=(9,3)); ax.axis('off'); tbl = ax.table(cellText=top3.values, colLabels=top3.columns, loc='center', cellLoc='center'); tbl.scale(1,2); ax.set_title('Top 3 Best-Selling Products', weight='bold');
    pdf.savefig(fig, bbox_inches='tight');
    plt.close()

print(top3)

        product_name Total Quantity Sold Total Revenue
0  cannelé bordelais              11,965    $55,457.02
1   pain de campagne               7,064    $32,487.58
2   macaron pistache               6,801    $31,045.62
